# ⚡ 05: Better RAG - Hybrid Search & Reranking

---

| 항목 | 내용 |
|------|------|
| **목표** | naive RAG의 한계를 보이고, hybrid retrieval로 검색 품질을 높이는 방법을 이해한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 7분 / 전체 18분 |
| **API 키** | ❌ 불필요 (로컬 임베딩 + BM25) |
| **이전 노트북과의 연결** | 기본 RAG를 만들었다. 이제 '왜 검색 실패가 곧 답변 실패인지', 어떻게 개선하는지 본다. |

---

## 🎯 핵심 메시지

> **2026년 기준, 중요한 것은 vector DB 자체가 아니라 retrieval quality입니다.**  
> **검색 실패 = 답변 실패.**

```
Keyword (BM25) : 정확한 단어 매칭에 강함
Vector Search  : 의미 이해에 강함
Hybrid         : 두 방식을 결합 → 더 강인한 검색
Reranking      : 검색된 결과를 다시 정렬 → precision 향상
```

In [ ]:
!pip install -q sentence-transformers rank-bm25 openai
print("✅ 완료")

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from IPython.display import display, HTML

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(key=""):
    k = key or os.environ.get("OPENAI_API_KEY", "")
    if k and k not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=k)
            print("✅ API 모드")
            return "api", c
        except: pass
    print("✅ 로컬 모드 (BM25 + sentence-transformers)")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

_st = None
def embed(texts):
    global _st
    if MODE == "api" and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                r = client.embeddings.create(input=texts[i:i+50], model="text-embedding-3-small")
                vecs.extend([x.embedding for x in r.data])
            return np.array(vecs, dtype=np.float32)
        except: pass
    if _st is None:
        print("📥 임베딩 모델 로딩...")
        from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅")
    return _st.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def cos_sim(q, D):
    q = np.array(q, dtype=np.float32).flatten()
    D = np.array(D, dtype=np.float32)
    qn = np.linalg.norm(q)
    if qn < 1e-9: return np.zeros(len(D))
    dn = np.linalg.norm(D, axis=1)
    dn = np.where(dn < 1e-9, 1e-9, dn)
    return (D @ q) / (dn * qn)

print(f"모드: {MODE}")

## 1️⃣ 데이터 준비 + 인덱스 구축

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# hybrid search 효과가 잘 드러나도록 설계된 문서셋
from helpers.sample_data import SAMPLE_DOCS_05 as DOCS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

# 벡터 인덱스
print("📊 벡터 인덱스 구축 중...")
contents = [d["content"] for d in DOCS]
doc_vecs = embed(contents)
print(f"✅ 벡터 완료: {doc_vecs.shape}")

# BM25 인덱스
def tokenize(text):
    tokens = re.sub(r'[^\w\s]', ' ', text).split()
    return [t.lower() for t in tokens if len(t) > 1]

tokenized = [tokenize(c) for c in contents]
bm25_idx = BM25Okapi(tokenized)
print("✅ BM25 인덱스 완료")

## 2️⃣ 세 가지 검색 방식 구현

In [ ]:
def keyword_only(query, top_k=5):
    """BM25 키워드 검색."""
    tokens = tokenize(query)
    scores = bm25_idx.get_scores(tokens)
    idx = np.argsort(scores)[::-1][:top_k]
    return [{**DOCS[i], "score": float(scores[i]), "method": "keyword"} for i in idx]


def vector_only(query, top_k=5):
    """벡터 시맨틱 검색."""
    q_vec = embed([query])[0]
    scores = cos_sim(q_vec, doc_vecs)
    idx = np.argsort(scores)[::-1][:top_k]
    return [{**DOCS[i], "score": float(scores[i]), "method": "vector"} for i in idx]


def hybrid(query, top_k=5, alpha=0.5):
    """
    하이브리드 검색: BM25 + Vector 결합.
    alpha=1.0 → 벡터만, alpha=0.0 → BM25만
    """
    # 각 방식의 점수 계산
    kw_scores = np.array([r["score"] for r in keyword_only(query, top_k=len(DOCS))])
    vec_scores_list = vector_only(query, top_k=len(DOCS))

    # 순서 맞추기 (keyword_only와 vector_only가 같은 순서로 DOCS를 반환하지 않을 수 있음)
    kw_tokens = tokenize(query)
    kw_raw = bm25_idx.get_scores(kw_tokens)
    q_vec = embed([query])[0]
    vec_raw = cos_sim(q_vec, doc_vecs)

    # Min-max 정규화
    def norm(arr):
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-9:
            return np.ones_like(arr) * 0.5
        return (arr - mn) / (mx - mn)

    kw_norm = norm(kw_raw)
    vec_norm = norm(vec_raw)

    combined = alpha * vec_norm + (1 - alpha) * kw_norm
    idx = np.argsort(combined)[::-1][:top_k]

    return [
        {
            **DOCS[i],
            "score": float(combined[i]),
            "vec_score": float(vec_norm[i]),
            "kw_score": float(kw_norm[i]),
            "method": "hybrid",
        }
        for i in idx
    ]


def compare_search_methods(query, top_k=5):
    """세 가지 방식의 검색 결과를 나란히 비교합니다."""
    kw_res = keyword_only(query, top_k)
    vec_res = vector_only(query, top_k)
    hyb_res = hybrid(query, top_k)

    print(f"\n{'='*62}")
    print(f"  쿼리: '{query}'")
    print(f"{'='*62}")

    headers = ["순위", "키워드(BM25)", "벡터(Semantic)", "하이브리드"]
    rows = []
    for i in range(top_k):
        kw = kw_res[i] if i < len(kw_res) else {"doc_id": "-", "score": 0}
        vc = vec_res[i] if i < len(vec_res) else {"doc_id": "-", "score": 0}
        hb = hyb_res[i] if i < len(hyb_res) else {"doc_id": "-", "score": 0}
        rows.append({
            "순위": f"#{i+1}",
            "키워드(BM25)": f"{kw['doc_id']} ({kw['score']:.3f})",
            "벡터(Semantic)": f"{vc['doc_id']} ({vc['score']:.4f})",
            "하이브리드": f"{hb['doc_id']} ({hb['score']:.4f})",
        })

    display(pd.DataFrame(rows))
    return kw_res, vec_res, hyb_res


print("✅ 검색 함수 준비 완료")

## 3️⃣ 케이스 A: 키워드 검색이 강한 경우

정확한 버전 번호, 기술 용어 → BM25가 유리합니다.

In [ ]:
# 정확한 기술 키워드 포함 쿼리
q_kw = "Python 3.10 버전 REST API v1 제거"
kw1, vec1, hyb1 = compare_search_methods(q_kw, top_k=4)

print("\n분석:")
print("  'Python 3.10', 'REST API v1' 같은 정확한 기술 키워드는")
print("  BM25가 정확히 해당 문서를 찾아냅니다.")
print(f"  → BM25 #1: {kw1[0]['doc_id']} (점수: {kw1[0]['score']:.3f})")
print(f"  → Vector #1: {vec1[0]['doc_id']} (점수: {vec1[0]['score']:.4f})")
print(f"  → Hybrid #1: {hyb1[0]['doc_id']} (점수: {hyb1[0]['score']:.4f})")

## 4️⃣ 케이스 B: 의미 검색이 강한 경우

개념적 질문, 맥락 이해 필요 → Vector Search가 유리합니다.

In [ ]:
# 의미 이해가 필요한 쿼리
q_sem = "비밀 정보를 소스코드에 넣으면 안 되는 이유와 올바른 방법"
kw2, vec2, hyb2 = compare_search_methods(q_sem, top_k=4)

print("\n분석:")
print("  '비밀 정보', '소스코드에 넣으면 안 되는' → 실제 문서에는")
print("  '시크릿', '하드코딩 금지'라는 표현이 사용됨.")
print("  BM25는 단어 불일치로 실패할 수 있지만,")
print("  Vector Search는 의미적 유사성으로 올바른 문서를 찾습니다.")
print(f"  → BM25 #1: {kw2[0]['doc_id']}")
print(f"  → Vector #1: {vec2[0]['doc_id']} ← 보안 정책 문서여야 함")
print(f"  → Hybrid #1: {hyb2[0]['doc_id']}")

## 5️⃣ 케이스 C: 하이브리드의 진가

키워드와 의미가 모두 중요한 쿼리.

In [ ]:
# 두 가지 모두 필요한 쿼리
q_hybrid = "DataPulse v1.1 업그레이드 후 메모리 설정과 알림 설정 방법"
kw3, vec3, hyb3 = compare_search_methods(q_hybrid, top_k=5)

print("\n각 방식 #1 결과 비교:")
print(f"  BM25: {kw3[0]['doc_id']} - {kw3[0]['content'][:60]}...")
print(f"  Vector: {vec3[0]['doc_id']} - {vec3[0]['content'][:60]}...")
print(f"  Hybrid: {hyb3[0]['doc_id']} - {hyb3[0]['content'][:60]}...")
print()
print("  상위 3개에 REL-DATAPULSE-011, FAQ-DATAPULSE-001이 모두 있는지 확인:")
target_docs = {"REL-DATAPULSE-011", "FAQ-DATAPULSE-001"}
for method, results in [("BM25", kw3), ("Vector", vec3), ("Hybrid", hyb3)]:
    found = {r["doc_id"] for r in results[:3]} & target_docs
    print(f"  {method}: {found} {'✅' if len(found) >= 2 else '⚠️ 일부 누락'}")

## 6️⃣ Alpha 파라미터 실험 - 혼합 비율 조정

In [ ]:
# Alpha 값에 따른 검색 결과 변화
q_test = "API 인증 방식 변경 v2.3"
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]

print(f"쿼리: '{q_test}'")
print(f"Alpha 실험 (0=키워드만, 1=벡터만):")
print()

rows = []
for a in alphas:
    results = hybrid(q_test, top_k=3, alpha=a)
    top3_ids = ", ".join(r["doc_id"] for r in results)
    desc = {0.0: "BM25 only", 0.3: "BM25 우선", 0.5: "균형", 0.7: "Vector 우선", 1.0: "Vector only"}[a]
    rows.append({"Alpha": a, "설명": desc, "상위 3개 문서": top3_ids})

display(pd.DataFrame(rows))

print()
print("💡 실제 시스템에서는")
print("   - 기술 키워드 많은 도메인: alpha 낮게 (BM25 비중 높게)")
print("   - 의미 이해 중심 도메인: alpha 높게 (Vector 비중 높게)")
print("   - 일반적으로 0.4~0.6이 좋은 출발점")

## 7️⃣ Reranking 개념 시연

검색 이후에 결과를 다시 정렬하는 Reranking 패턴을 보여줍니다.

In [ ]:
def simple_rerank(query, candidates, method="cross_score"):
    """
    간단한 Reranking 시뮬레이션.
    실제 서비스에서는 cross-encoder 모델(ms-marco 등)을 사용합니다.
    여기서는 교육용으로 간단한 관련성 점수를 계산합니다.
    """
    query_tokens = set(tokenize(query))

    scored = []
    for doc in candidates:
        # 간단한 관련성 점수: 쿼리 토큰 중 문서에 있는 비율
        doc_tokens = set(tokenize(doc["content"]))
        overlap = len(query_tokens & doc_tokens)
        coverage = overlap / max(len(query_tokens), 1)

        # 문서 최신성 보너스 (더 최신 문서 우선)
        recency_bonus = 0.1 if "v3.1" in doc["content"] or "2024-11" in doc.get("doc_id", "") else 0

        # 구버전 패널티
        old_version_penalty = -0.2 if "구버전" in doc["content"] or "비효력" in doc["content"] else 0

        rerank_score = doc["score"] + (coverage * 0.2) + recency_bonus + old_version_penalty

        scored.append({
            **doc,
            "original_score": doc["score"],
            "rerank_score": rerank_score,
            "coverage": coverage,
        })

    return sorted(scored, key=lambda x: x["rerank_score"], reverse=True)


# Reranking 효과 시연: 구버전 vs 현행 문서
q_rerank = "API 키 만료 기간은 얼마인가요?"

# 1. 기본 검색
candidates = hybrid(q_rerank, top_k=5)

# 2. Reranking
reranked = simple_rerank(q_rerank, candidates)

print(f"쿼리: '{q_rerank}'")
print()
print("검색 전 순위 vs Reranking 후 순위:")
print()

rows = []
for i, (before, after) in enumerate(zip(candidates, reranked), 1):
    old_flag = "⚠️구버전" if "구버전" in before["content"] or "비효력" in before["content"] else ""
    rows.append({
        "원래 순위": f"#{i} {before['doc_id']}{old_flag}",
        "원래 점수": f"{before['score']:.4f}",
        "Rerank 후": f"#{i} {after['doc_id']}",
        "Rerank 점수": f"{after['rerank_score']:.4f}",
    })

display(pd.DataFrame(rows))

print()
print("💡 Reranking 포인트:")
print("   - 구버전 문서(비효력)에 패널티 적용")
print("   - 쿼리 토큰 커버리지가 높은 문서에 보너스")
print("   - 실제 서비스: cross-encoder 모델 (ms-marco-MiniLM 등) 사용")

## 8️⃣ 검색 실패 → 답변 실패 확인

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 검색 방식에 따라 답변 품질이 얼마나 달라지는지 정량적으로 보여주기
from helpers.sample_data import TEST_CASES_05 as test_cases
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

def eval_retrieval(query, relevant_doc_ids, top_k=3):
    """검색 방식별 Precision@k를 계산합니다."""
    results = {
        "BM25": keyword_only(query, top_k),
        "Vector": vector_only(query, top_k),
        "Hybrid": hybrid(query, top_k),
    }

    print(f"\n쿼리: '{query}'")
    print(f"정답 문서: {relevant_doc_ids}")
    print()

    for method, res in results.items():
        found_ids = [r["doc_id"] for r in res]
        hits = sum(1 for did in found_ids if did in relevant_doc_ids)
        precision = hits / top_k
        bar = "█" * hits + "░" * (top_k - hits)
        print(f"  {method:8s}: [{bar}] Precision@{top_k}={precision:.2f}  검색된 문서: {found_ids}")


for query, relevant in test_cases:
    eval_retrieval(query, relevant, top_k=3)
    print()

In [ ]:
# Retrieval 전략 선택 가이드 시각화
display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:800px;margin:10px auto;">
  <h3 style="color:#2c3e50;">Retrieval 전략 선택 가이드</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">전략</th>
        <th style="padding:10px;">장점</th>
        <th style="padding:10px;">단점</th>
        <th style="padding:10px;">적합한 상황</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;"><strong>BM25</strong></td>
        <td style="padding:10px;">빠름, 설명 가능, 정확한 키워드</td>
        <td style="padding:10px;">동의어 처리 불가</td>
        <td style="padding:10px;">코드, 버전, 제품명 검색</td>
      </tr>
      <tr>
        <td style="padding:10px;"><strong>Vector</strong></td>
        <td style="padding:10px;">의미 이해, 다국어</td>
        <td style="padding:10px;">비용, 정확한 키워드 약함</td>
        <td style="padding:10px;">개념 검색, Q&A, 고객 문의</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;"><strong>Hybrid</strong></td>
        <td style="padding:10px;">두 방식의 장점 결합</td>
        <td style="padding:10px;">구현 복잡도 증가</td>
        <td style="padding:10px;">대부분의 실제 서비스 (권장)</td>
      </tr>
      <tr>
        <td style="padding:10px;"><strong>Hybrid + Rerank</strong></td>
        <td style="padding:10px;">최고 precision</td>
        <td style="padding:10px;">레이턴시 증가, 비용</td>
        <td style="padding:10px;">정확도가 매우 중요한 도메인</td>
      </tr>
    </tbody>
  </table>
  <div style="margin-top:12px;padding:12px;background:#fff9c4;border-left:4px solid #f39c12;font-size:13px;">
    ⚡ <strong>2026년 실전 기준</strong>: 대부분의 production RAG는 Hybrid + Reranking을 기본으로 씁니다.<br>
    Vector DB 선택보다 <strong>어떤 검색 전략을 쓸지</strong>가 더 중요합니다.
  </div>
</div>
"""))

---

## 🎤 강의자 멘트 포인트

> **"Vector DB 제품을 고르는 것보다 중요한 질문은 '어떤 검색 전략을 쓸 것인가'입니다.**
>
> 벡터 검색만 쓰면 '시크릿 관리'와 '하드코딩 금지'가 같은 뜻인 걸 알지만,  
> 'Python 3.10'이라는 정확한 버전은 잡을 수 없을 수 있습니다.  
>
> 반대로 키워드만 쓰면 '보안 정책'을 물어봐도  
> '관리', '정책' 단어 가 없는 문서는 아예 못 찾습니다.  
>
> Hybrid가 기본입니다. 비용과 복잡도가 걱정되면  
> 우선 BM25로 시작해서 Vector를 얹는 방향으로 가세요."

## 🙋 청중 질문 유도
> - "여러분 서비스에서는 키워드 검색과 의미 검색 중 어느 게 더 중요할까요?"
> - "Reranking에서 '구버전 문서 패널티'를 어떻게 자동으로 걸 수 있을까요?"
> - "검색 품질을 어떻게 측정하나요? 어떤 지표를 쓰나요? (Precision, NDCG, MRR)"

## 🏗️ 실무 확장 포인트
- **Cross-encoder Reranking**: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **Query Expansion**: LLM으로 쿼리를 확장하여 recall 향상
- **Metadata Filtering**: 날짜, 카테고리, 버전으로 검색 범위 제한
- **Evaluation Framework**: RAGAS, TruLens로 retrieval quality 자동 측정

## ➕ 추가 실험
1. `alpha` 값을 0.0~1.0으로 바꿔가며 각 케이스에서 최적값 찾기
2. 구버전 SEC-POL-002를 제거하면 검색 품질이 어떻게 달라지는지 확인
3. 쿼리에 의도적인 오타를 넣어 각 방식의 내성 비교

## ➡️ 다음 노트북
**06_agentic_search_or_tool_use.ipynb** - 한 번 검색으로 끝내지 않고, 모델이 도구를 반복 사용하는 방식